# Analyzing Mobile App Data

We only build apps that are free to download and install, and our main source of revenue consists of in-app ads. This means that the number of users of our apps determines our revenue for any given app — the more users who see and engage with the ads, the better. Our goal for this project is to analyze data to help our developers understand what type of apps are likely to attract more users.

Our datasets can be found here:\
[Apple Store Dataset](https://www.kaggle.com/datasets/ramamet4/app-store-apple-data-set-10k-apps)\
[Google Play Store](https://www.kaggle.com/datasets/lava18/google-play-store-apps)

First we are going to open them:

In [1]:
from csv import reader

opened_file = open("AppleStore.csv", encoding="utf-8")
read_file = reader(opened_file)
apple = list(read_file)
apple_data = apple[1:]
apple_header = apple[0]

opened_file = open("googleplaystore.csv", encoding="utf-8")
read_file = reader(opened_file)
google = list(read_file)
google_data = google[1:]
google_header = google[0]

## Exploring the Data

First we are going to take a dive into the data to see what we are working with! To do that we start with a function that will let us do just that.

In [2]:
def explore_data(dataset, start, end, rows_and_columns=False):
    dataset_slice = dataset[start:end]
    for row in dataset_slice:
        print(row)
        print("\n")

    if rows_and_columns:
        print(f"Number of rows: {len(dataset)}")
        print(f"Number of columns: {len(dataset[0])}")
        

### Apple Store

In [3]:
explore_data(apple_data, 0, 5, True)

['284882215', 'Facebook', '389879808', 'USD', '0.0', '2974676', '212', '3.5', '3.5', '95.0', '4+', 'Social Networking', '37', '1', '29', '1']


['389801252', 'Instagram', '113954816', 'USD', '0.0', '2161558', '1289', '4.5', '4.0', '10.23', '12+', 'Photo & Video', '37', '0', '29', '1']


['529479190', 'Clash of Clans', '116476928', 'USD', '0.0', '2130805', '579', '4.5', '4.5', '9.24.12', '9+', 'Games', '38', '5', '18', '1']


['420009108', 'Temple Run', '65921024', 'USD', '0.0', '1724546', '3842', '4.5', '4.0', '1.6.2', '9+', 'Games', '40', '5', '1', '1']


['284035177', 'Pandora - Music & Radio', '130242560', 'USD', '0.0', '1126879', '3594', '4.0', '4.5', '8.4.1', '12+', 'Music', '37', '4', '1', '1']


Number of rows: 7197
Number of columns: 16


Now we know that there are 7,197 rows and 16 columns in this dataset.

Let's also print the columns of the dataset to get to know it.

In [4]:
print(apple_header)

['id', 'track_name', 'size_bytes', 'currency', 'price', 'rating_count_tot', 'rating_count_ver', 'user_rating', 'user_rating_ver', 'ver', 'cont_rating', 'prime_genre', 'sup_devices.num', 'ipadSc_urls.num', 'lang.num', 'vpp_lic']


### Google Play Store

In [5]:
explore_data(google_data, 0, 5, True)

['Photo Editor & Candy Camera & Grid & ScrapBook', 'ART_AND_DESIGN', '4.1', '159', '19M', '10,000+', 'Free', '0', 'Everyone', 'Art & Design', 'January 7, 2018', '1.0.0', '4.0.3 and up']


['Coloring book moana', 'ART_AND_DESIGN', '3.9', '967', '14M', '500,000+', 'Free', '0', 'Everyone', 'Art & Design;Pretend Play', 'January 15, 2018', '2.0.0', '4.0.3 and up']


['U Launcher Lite – FREE Live Cool Themes, Hide Apps', 'ART_AND_DESIGN', '4.7', '87510', '8.7M', '5,000,000+', 'Free', '0', 'Everyone', 'Art & Design', 'August 1, 2018', '1.2.4', '4.0.3 and up']


['Sketch - Draw & Paint', 'ART_AND_DESIGN', '4.5', '215644', '25M', '50,000,000+', 'Free', '0', 'Teen', 'Art & Design', 'June 8, 2018', 'Varies with device', '4.2 and up']


['Pixel Draw - Number Art Coloring Book', 'ART_AND_DESIGN', '4.3', '967', '2.8M', '100,000+', 'Free', '0', 'Everyone', 'Art & Design;Creativity', 'June 20, 2018', '1.1', '4.4 and up']


Number of rows: 10841
Number of columns: 13


Now we know that there are 10,841 rows and 13 columns in this dataset.

Let's print the columns of the dataset.

In [6]:
print(google_header)

['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver']


## Cleaning data

Now we are going to clean the data, removing non-English apps, apps that aren't free and apps that have something wrong in its record.

In the [discussion section](https://www.kaggle.com/datasets/lava18/google-play-store-apps/discussion?sort=undefined) of the Google Play dataset there is [a discussion](https://www.kaggle.com/datasets/lava18/google-play-store-apps/discussion/66015) about a row with a wrong value in a column.

The value of this row is 10472. Let's see what is happening.

In [7]:
print(google_data[10472])

['Life Made WI-Fi Touchscreen Photo Frame', '1.9', '19', '3.0M', '1,000+', 'Free', '0', 'Everyone', '', 'February 11, 2018', '1.0.19', '4.0 and up']


We can see that in the third column the value for rating is 19. This is impossible. What probably happened is that there is a shift in the column values of this app ('1.9' seems a plausible value for the rating). As we can't be sure, we will delete it from our analysis.

In [8]:
del google_data[10472]

Now we are going to remove duplicates from the Google Play Store database (the Apple Store database does not have duplicates). First, let's see how many duplicates are we talking about!

In [9]:
duplicate_apps = []
existing_apps = []

for app in google_data:
    if app[0] in existing_apps:
        duplicate_apps.append(app[0])
    else:
        existing_apps.append(app[0])

print(f"Number of duplicate apps: {len(duplicate_apps)}")

Number of duplicate apps: 1181


Some of the apps that are duplicated appear more than two times in our list. We can't choose one of the records at random to delete it.

If we look at our data we can see that the number of reviews column have different values for the duplicated apps. If we take the one with the heighest number it will probably be the last time the app was recorded in our database. This seems a good path for us to clean the duplicates - we mantain only the one record with the heighest number of reviews.

In [10]:
reviews_max = {}

for app in google_data:
    name = app[0]
    review_count = app[3]
    if name in reviews_max and review_count > reviews_max[name]:
        reviews_max[name] = review_count
    elif name not in reviews_max:
        reviews_max[name] = review_count

print(len(reviews_max))

9659


Now let's remove the duplicated ones:

In [11]:
google_clean = []
already_added = []

for app in google_data:
    name = app[0]
    rating_count = app[3]
    if rating_count == reviews_max[name] and name not in already_added:
        already_added.append(name)
        google_clean.append(app)

In [12]:
print(len(google_clean))

9659


### Removing non-English apps

Let's first construct a function that will return False if there's any character in the string that doesn't belong to the set of common English characters; otherwise, the funcion returns True.

In [13]:
def is_english(string):
    for char in string:
        if ord(char) < 0 or ord(char) > 127:
            return False
    return True

Let's test our function:

In [14]:
print(is_english('Instagram'))
print(is_english('爱奇艺PPS -《欢乐颂2》电视剧热播'))
print(is_english('Docs To Go™ Free Office Suite'))
print(is_english('Instachat 😜'))

True
False
False
False


There's a problem with our method: the third and last tests are in English. They include a char that's outside of the ASCII range that we included.

To mitigate this risk, we are going to bar apps with at least 3 non-ASCII complient chars. It's not ideal, but we can be sure that it is good enough.

In [15]:
def is_english(string):
    count_non_ascii = 0
    for char in string:
        if ord(char) < 0 or ord(char) > 127:
            count_non_ascii += 1
            if count_non_ascii > 3:
                return False
    return True

Let's test it again:

In [16]:
print(is_english('Instagram'))
print(is_english('爱奇艺PPS -《欢乐颂2》电视剧热播'))
print(is_english('Docs To Go™ Free Office Suite'))
print(is_english('Instachat 😜'))

True
False
True
True


Now we are going to use this function with our datasets!

In [17]:
apple_clean_en = []

for app in apple_data:
    if is_english(app[1]):
        apple_clean_en.append(app)

print(len(apple_clean_en))

6183


In [18]:
google_clean_en = []

for app in google_clean:
    if is_english(app[0]):
        google_clean_en.append(app)

print(len(google_clean_en))

9614


We are left with 6,183 Apple Store apps and 9,614 Google Play Store apps!

### Removing non-free apps

Our company is mainly focused on free apps. Because of this we are going to remove the non-free apps from our analysis!

In [19]:
apple_clean_en_free = []

for app in apple_clean_en:
    price = app[4]
    if price == "0.0":
        apple_clean_en_free.append(app)

print(len(apple_clean_en_free))

3222


In [20]:
google_clean_en_free = []

for app in google_clean_en:
    price = app[7]
    if price == "0":
        google_clean_en_free.append(app)

print(len(google_clean_en_free))

8862


We are left with 3,222 Apple Store apps and 8,862 Google Play Store apps!

## Generating genres frequency tables

Our company has a strategy:
1. Build a minimal Android version of the app, and add it to Google Play
2. If the app has a good response from users, we develop it further
3. If the app is profitable after six months, we build an iOS version of the app and add it to the App Store.

Our end goal is to launch the app both on Google Play and App Store, so we need to find app profiles that are successful in both markets.

To do this we will determine the most common genres for each market. To do this we will generate frequency tables!

First we construct a function that generates frequency tables.

In [21]:
def freq_table(dataset, index):
    frequency_table = {}
    for data in dataset:
        if data[index] not in frequency_table:
            frequency_table[data[index]] = 1
        else:
            frequency_table[data[index]] += 1
    return frequency_table
            

Then we build a function that will display them in reverse order:

In [22]:
def display_table(dataset, index):
    table = freq_table(dataset, index)
    table_display = []
    for key in table:
        key_val_as_tuple = (table[key], key)
        table_display.append(key_val_as_tuple)

    table_sorted = sorted(table_display, reverse=True)
    for entry in table_sorted:
        print(entry[1], ":", entry[0])

Now lets see it applied to the Apple Store dataset: 

In [23]:
display_table(apple_clean_en_free, 11)

Games : 1874
Entertainment : 254
Photo & Video : 160
Education : 118
Social Networking : 106
Shopping : 84
Utilities : 81
Sports : 69
Music : 66
Health & Fitness : 65
Productivity : 56
Lifestyle : 51
News : 43
Travel : 40
Finance : 36
Weather : 28
Food & Drink : 26
Reference : 18
Business : 17
Book : 14
Navigation : 6
Medical : 6
Catalogs : 4


Interesting, Games are the vast majority of the apps in the Apple Store, followed by Entertainment ones and then Photo & Video.

Now lets do it to the Google Play Store category and genres:

In [24]:
display_table(google_clean_en_free, 1)

FAMILY : 1678
GAME : 859
TOOLS : 749
BUSINESS : 407
LIFESTYLE : 346
PRODUCTIVITY : 345
FINANCE : 328
MEDICAL : 312
SPORTS : 301
PERSONALIZATION : 294
COMMUNICATION : 287
HEALTH_AND_FITNESS : 273
PHOTOGRAPHY : 261
NEWS_AND_MAGAZINES : 248
SOCIAL : 236
TRAVEL_AND_LOCAL : 207
SHOPPING : 199
BOOKS_AND_REFERENCE : 190
DATING : 165
VIDEO_PLAYERS : 159
MAPS_AND_NAVIGATION : 124
FOOD_AND_DRINK : 110
EDUCATION : 104
ENTERTAINMENT : 85
LIBRARIES_AND_DEMO : 83
AUTO_AND_VEHICLES : 82
HOUSE_AND_HOME : 73
WEATHER : 71
EVENTS : 63
PARENTING : 58
ART_AND_DESIGN : 57
COMICS : 55
BEAUTY : 53


In [25]:
display_table(google_clean_en_free, 9)

Tools : 748
Entertainment : 538
Education : 474
Business : 407
Productivity : 345
Lifestyle : 345
Finance : 328
Medical : 312
Sports : 307
Personalization : 294
Communication : 287
Action : 275
Health & Fitness : 273
Photography : 261
News & Magazines : 248
Social : 236
Travel & Local : 206
Shopping : 199
Books & Reference : 190
Simulation : 181
Dating : 165
Arcade : 164
Video Players & Editors : 157
Casual : 155
Maps & Navigation : 124
Food & Drink : 110
Puzzle : 100
Racing : 88
Role Playing : 83
Libraries & Demo : 83
Auto & Vehicles : 82
Strategy : 81
House & Home : 73
Weather : 71
Events : 63
Adventure : 60
Comics : 54
Beauty : 53
Art & Design : 53
Parenting : 44
Card : 39
Casino : 38
Trivia : 37
Educational;Education : 35
Educational : 33
Board : 33
Education;Education : 30
Word : 23
Casual;Pretend Play : 21
Music : 18
Puzzle;Brain Games : 16
Racing;Action & Adventure : 15
Entertainment;Music & Video : 15
Casual;Brain Games : 12
Casual;Action & Adventure : 12
Arcade;Action & Advent

On the Google Play Store there are more tools focused apps, but games are pretty huge too!

For now, we know which kinds of apps are most proiminent on the stores. That doesn't mean that most users are using them.

Luckly for us the Google Play Store dataset provides us with a 'Installs" column - that will be handy! The Apple Store doesn't provide us with one, but we can use number of user ratings as a proxy - that's the closest we will get to the exact number of installs on the Apple Store.

In [26]:
prime_genre_freq_table = freq_table(apple_clean_en_free, 11)

for genre in prime_genre_freq_table:
    total = 0
    len_genre = 0
    for app in apple_clean_en_free:
        genre_app = app[11]
        if genre == genre_app:
            number_user_ratings = float(app[5])
            total += number_user_ratings
            len_genre += 1
    avg_n_user_rating = total / len_genre
    print(f"{genre} - {avg_n_user_rating}")

Social Networking - 71548.34905660378
Photo & Video - 28441.54375
Games - 22788.6696905016
Music - 57326.530303030304
Reference - 74942.11111111111
Health & Fitness - 23298.015384615384
Weather - 52279.892857142855
Utilities - 18684.456790123455
Travel - 28243.8
Shopping - 26919.690476190477
News - 21248.023255813954
Navigation - 86090.33333333333
Lifestyle - 16485.764705882353
Entertainment - 14029.830708661417
Food & Drink - 33333.92307692308
Sports - 23008.898550724636
Book - 39758.5
Finance - 31467.944444444445
Education - 7003.983050847458
Productivity - 21028.410714285714
Business - 7491.117647058823
Catalogs - 4004.0
Medical - 612.0


According to our numbers Navigation, Reference, Social Networking are the most used apps in the Apple Store!

Waze and Google Maps are bringing the Navigation number of ratings up! The same applies to Social Networking apps (Facebook, Skype, Instagram, etc). Bible and Dictionary is skewing the Reference numbers.

Lets look at the Google Store numbers. The install column doesn't bring us precise numbers - it brings us an interval. For our purposes we will use the numbers as they are. To do this we have to transform them in floats!

In [27]:
prime_category_freq_table = freq_table(google_clean_en_free, 1)
for genre in prime_category_freq_table:
    total = 0
    len_category = 0
    for app in google_clean_en_free:
        category_app = app[1]
        if category_app == genre:
            number_of_installs = float(app[5].replace("+", "").replace(",", ""))
            total += number_of_installs
            len_category += 1
    avg_category_install = total / len_category
    print(f"{genre} - {avg_category_install}")
            

ART_AND_DESIGN - 1986335.0877192982
AUTO_AND_VEHICLES - 647317.8170731707
BEAUTY - 513151.88679245283
BOOKS_AND_REFERENCE - 8767811.894736841
BUSINESS - 1712290.1474201474
COMICS - 817657.2727272727
COMMUNICATION - 38456119.167247385
DATING - 854028.8303030303
EDUCATION - 1820673.076923077
ENTERTAINMENT - 11640705.88235294
EVENTS - 253542.22222222222
FINANCE - 1387692.475609756
FOOD_AND_DRINK - 1924897.7363636363
HEALTH_AND_FITNESS - 4188821.9853479853
HOUSE_AND_HOME - 1331540.5616438356
LIBRARIES_AND_DEMO - 638503.734939759
LIFESTYLE - 1437816.2687861272
GAME - 15560965.599534342
FAMILY - 3694276.334922527
MEDICAL - 120616.48717948717
SOCIAL - 23253652.127118643
SHOPPING - 7036877.311557789
PHOTOGRAPHY - 17805627.643678162
SPORTS - 3638640.1428571427
TRAVEL_AND_LOCAL - 13984077.710144928
TOOLS - 10682301.033377837
PERSONALIZATION - 5201482.6122448975
PRODUCTIVITY - 16787331.344927534
PARENTING - 542603.6206896552
WEATHER - 5074486.197183099
VIDEO_PLAYERS - 24727872.452830188
NEWS_AND_

Books and Reference seem to be a big one on Google Play Store too. 

## Conclusion

Based on this results I would recommend to our company that we launched a famous book as an app on the Google Store. We could launch some other functionalities to the app, like an audio book version, quotes, discussion forum